# 08 — Permutation Feature Importance for Full Hybrid Model

Global feature importance using **permutation importance** from sklearn on the entire hybrid ensemble.

**How it works:**
1. Compute baseline metric (e.g., MAPE) on test set
2. For each feature, shuffle its values and re-compute metric
3. Importance = increase in error when feature is shuffled

**Prereqs:** Run **02_hybrid_ensemble.ipynb** so `03_ml_layer_hybrid/artifacts/hybrid_cluster_bundle.joblib` exists.

**Trade-offs:**
- **Pros:** Works on any model (black-box), captures full hybrid behavior
- **Cons:** Computationally expensive, gives global importance only (no per-prediction breakdown)

**Outputs:** `03_ml_layer_hybrid/artifacts/hybrid_xai/permutation_importance_*.json`

In [1]:
%pip install -q numpy pandas scikit-learn xgboost lightgbm matplotlib joblib

Note: you may need to restart the kernel to use updated packages.


In [2]:
import importlib
import json
import time
import warnings
from pathlib import Path

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_percentage_error, make_scorer

warnings.filterwarnings("ignore")

import sys

_HERE = Path.cwd().resolve()


def _repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "data" / "feature_data").is_dir() or (p / "hf_data").is_dir():
            return p
    return start.parent


REPO_ROOT = _repo_root(_HERE)

_FEATURE_OUTPUT_CANDIDATES = [
    REPO_ROOT / "hf_data" / "02_feature_layer" / "training" / "outputs",
    REPO_ROOT / "data" / "feature_data" / "02_feature_layer" / "training" / "outputs",
]


def _feature_outputs_dir() -> Path:
    for d in _FEATURE_OUTPUT_CANDIDATES:
        if d.is_dir() and any(d.glob("hdb_feature_table_*.csv")):
            return d
    tried = "\n  ".join(str(d) for d in _FEATURE_OUTPUT_CANDIDATES)
    raise FileNotFoundError(
        "No hdb_feature_table_*.csv found. Run notebooks/00_download_data_from_HF.ipynb "
        "or place CSVs under one of:\n  " + tried
    )


def _hybrid_ml_dir(repo: Path) -> Path:
    candidates = [
        repo / "notebooks" / "03_ml_layer_hybrid",
        repo / "03_ml_layer_hybrid",
    ]
    for d in candidates:
        if (d / "yc_hybrid_inference.py").is_file():
            return d
    raise FileNotFoundError(
        "yc_hybrid_inference.py not found. Expected under notebooks/03_ml_layer_hybrid/."
    )


HYBRID_DIR = _hybrid_ml_dir(REPO_ROOT)
if str(HYBRID_DIR) not in sys.path:
    sys.path.insert(0, str(HYBRID_DIR))

import yc_hybrid_inference
importlib.reload(yc_hybrid_inference)
from yc_hybrid_inference import predict_price, load_bundle

HF_DATA_ROOT = _feature_outputs_dir()


def _latest_feature_snapshot_date(root: Path) -> str:
    tables = sorted(root.glob("hdb_feature_table_*.csv"))
    if not tables:
        raise FileNotFoundError(f"No hdb_feature_table_*.csv under {root}")
    return tables[-1].stem.split("_")[-1]


_snap = _latest_feature_snapshot_date(HF_DATA_ROOT)
all_path = HF_DATA_ROOT / f"hdb_feature_table_{_snap}.csv"

HERE = REPO_ROOT
ART = HYBRID_DIR / "artifacts"
OUT_DIR = ART / "hybrid_xai"
OUT_DIR.mkdir(parents=True, exist_ok=True)
BUNDLE_PATH = ART / "hybrid_cluster_bundle.joblib"

TARGET = "resale_price"
YEAR_COL = "transaction_year"

print("HF_DATA_ROOT:", HF_DATA_ROOT)
print("Feature table:", all_path.name)
print("Artifacts:", ART)
print("OUT_DIR:", OUT_DIR)

HF_DATA_ROOT: /Users/bhuvesh/Documents/PropertyLens/data/feature_data/02_feature_layer/training/outputs
Feature table: hdb_feature_table_20260412.csv
Artifacts: /Users/bhuvesh/Documents/PropertyLens/notebooks/03_ml_layer_hybrid/artifacts
OUT_DIR: /Users/bhuvesh/Documents/PropertyLens/notebooks/03_ml_layer_hybrid/artifacts/hybrid_xai


## 1 — Load data and hybrid bundle

In [3]:
df = pd.read_csv(all_path)
df = df.sort_values([YEAR_COL, "address_key"], kind="mergesort").reset_index(drop=True)

bundle = joblib.load(BUNDLE_PATH)
FEATURE_COLS = bundle["feature_columns"]
N_CLUSTERS = int(bundle["n_clusters"])

train_mask = df[YEAR_COL] < 2024
val_mask = df[YEAR_COL] == 2024
test_mask = df[YEAR_COL] >= 2025

X_all = df[FEATURE_COLS].fillna(0).astype(float)
y_all = df[TARGET].astype(float)

X_train = X_all.loc[train_mask].values
y_train = y_all.loc[train_mask].values
X_test = X_all.loc[test_mask].values
y_test = y_all.loc[test_mask].values

print("Train rows:", len(X_train), "| Test rows:", len(X_test), "| Features:", len(FEATURE_COLS))

Train rows: 205930 | Test rows: 29266 | Features: 70


## 2 — Create sklearn-compatible wrapper

`sklearn.inspection.permutation_importance` requires an estimator with a `.predict()` method.

In [4]:
class HybridModelWrapper:
    """
    Sklearn-compatible wrapper for the hybrid ensemble model.
    
    Provides `.predict()` method required by sklearn's permutation_importance.
    """
    
    def __init__(self, bundle):
        self.bundle = bundle
    
    def predict(self, X):
        """Predict resale prices for feature matrix X."""
        return predict_price(np.asarray(X, dtype=float), self.bundle)
    
    def fit(self, X, y):
        """No-op fit method for sklearn compatibility."""
        return self


# Create wrapper
hybrid_wrapper = HybridModelWrapper(bundle)

# Verify it works
_test_pred = hybrid_wrapper.predict(X_test[:10])
_mape = mean_absolute_percentage_error(y_test[:10], _test_pred) * 100
print(f"Wrapper test - MAPE on 10 samples: {_mape:.2f}%")

Wrapper test - MAPE on 10 samples: 4.79%


## 3 — Compute permutation importance

We use a subsample of test data to reduce computation time. Multiple scorers are used to understand importance from different perspectives.

In [5]:
# Subsample test data for speed (permutation importance is expensive)
N_TEST_SAMPLES = min(3000, len(X_test))
N_REPEATS = 5

rng = np.random.RandomState(42)
test_idx = rng.choice(len(X_test), N_TEST_SAMPLES, replace=False)
X_test_sub = X_test[test_idx]
y_test_sub = y_test[test_idx]

print(f"Using {N_TEST_SAMPLES} test samples with {N_REPEATS} repeats per feature")
print(f"Total permutation evaluations: {len(FEATURE_COLS) * N_REPEATS} = {len(FEATURE_COLS)} features × {N_REPEATS} repeats")
print("-" * 50)

Using 3000 test samples with 5 repeats per feature
Total permutation evaluations: 350 = 70 features × 5 repeats
--------------------------------------------------


In [6]:
# Custom MAPE scorer (negative because sklearn maximizes)
def mape_scorer(y_true, y_pred):
    return -mean_absolute_percentage_error(y_true, y_pred) * 100

mape_score = make_scorer(mape_scorer, greater_is_better=True)

print("Computing permutation importance with MAPE scoring...")
t0 = time.time()

result_mape = permutation_importance(
    hybrid_wrapper,
    X_test_sub,
    y_test_sub,
    n_repeats=N_REPEATS,
    scoring=mape_score,
    n_jobs=-1,
    random_state=42,
)

elapsed = time.time() - t0
print(f"Completed in {elapsed:.1f}s ({elapsed/60:.1f} min)")

Computing permutation importance with MAPE scoring...


AttributeError: The following error was raised: 'HybridModelWrapper' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order.

In [ ]:
# Also compute with R² scoring for comparison
print("\nComputing permutation importance with R² scoring...")
t0 = time.time()

result_r2 = permutation_importance(
    hybrid_wrapper,
    X_test_sub,
    y_test_sub,
    n_repeats=N_REPEATS,
    scoring='r2',
    n_jobs=-1,
    random_state=42,
)

elapsed = time.time() - t0
print(f"Completed in {elapsed:.1f}s ({elapsed/60:.1f} min)")

## 4 — Analyze results

Feature importance = decrease in model performance when feature is shuffled.
Higher importance means the model relies more heavily on that feature.

In [ ]:
# Create importance DataFrames
importance_mape = pd.DataFrame({
    'feature': FEATURE_COLS,
    'importance_mean': result_mape.importances_mean,
    'importance_std': result_mape.importances_std,
}).sort_values('importance_mean', ascending=False)

importance_r2 = pd.DataFrame({
    'feature': FEATURE_COLS,
    'importance_mean': result_r2.importances_mean,
    'importance_std': result_r2.importances_std,
}).sort_values('importance_mean', ascending=False)

print("Top 15 features by permutation importance (MAPE scoring):")
print("(Higher = more important, values show increase in MAPE % when feature shuffled)")
print("-" * 70)
for i, row in importance_mape.head(15).iterrows():
    print(f"  {row['feature']:40s} {row['importance_mean']:>8.3f} ± {row['importance_std']:.3f}")

In [ ]:
print("\nTop 15 features by permutation importance (R² scoring):")
print("(Higher = more important, values show decrease in R² when feature shuffled)")
print("-" * 70)
for i, row in importance_r2.head(15).iterrows():
    print(f"  {row['feature']:40s} {row['importance_mean']:>8.4f} ± {row['importance_std']:.4f}")

## 5 — Visualizations

In [ ]:
# Bar plot of top features (MAPE scoring)
top_n = 20
top_features = importance_mape.head(top_n)

fig, ax = plt.subplots(figsize=(10, 8))
y_pos = np.arange(len(top_features))
ax.barh(y_pos, top_features['importance_mean'], xerr=top_features['importance_std'], 
        align='center', alpha=0.8, color='#2d7d6b')
ax.set_yticks(y_pos)
ax.set_yticklabels(top_features['feature'])
ax.invert_yaxis()
ax.set_xlabel('Permutation Importance (increase in MAPE %)')
ax.set_title('Hybrid Model - Permutation Feature Importance (MAPE)')
plt.tight_layout()
plt.savefig(OUT_DIR / "permutation_importance_mape.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved", OUT_DIR / "permutation_importance_mape.png")

In [ ]:
# Bar plot of top features (R² scoring)
top_features_r2 = importance_r2.head(top_n)

fig, ax = plt.subplots(figsize=(10, 8))
y_pos = np.arange(len(top_features_r2))
ax.barh(y_pos, top_features_r2['importance_mean'], xerr=top_features_r2['importance_std'], 
        align='center', alpha=0.8, color='#5b8db8')
ax.set_yticks(y_pos)
ax.set_yticklabels(top_features_r2['feature'])
ax.invert_yaxis()
ax.set_xlabel('Permutation Importance (decrease in R²)')
ax.set_title('Hybrid Model - Permutation Feature Importance (R²)')
plt.tight_layout()
plt.savefig(OUT_DIR / "permutation_importance_r2.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved", OUT_DIR / "permutation_importance_r2.png")

## 6 — Compare with SHAP importance

Load SHAP-based importance from previous notebooks and compare rankings.

In [ ]:
# Try to load SHAP importance from previous notebooks
shap_importance = None
kernel_shap_path = OUT_DIR / "kernel_shap_global_importance.json"
composite_shap_path = OUT_DIR / "composite_treeshap_global_importance.json"

if composite_shap_path.exists():
    with open(composite_shap_path) as f:
        data = json.load(f)
        shap_importance = data.get("global_feature_importance", {})
        print("Loaded SHAP importance from Composite TreeSHAP")
elif kernel_shap_path.exists():
    with open(kernel_shap_path) as f:
        data = json.load(f)
        shap_importance = data.get("feature_importance", {})
        print("Loaded SHAP importance from KernelSHAP")
else:
    print("No SHAP importance files found - run notebooks 06 or 07 first")
    print("Skipping comparison...")

In [ ]:
if shap_importance:
    # Create comparison DataFrame
    perm_rank = {f: i for i, f in enumerate(importance_mape['feature'].values)}
    shap_sorted = sorted(shap_importance.items(), key=lambda x: x[1], reverse=True)
    shap_rank = {f: i for i, (f, _) in enumerate(shap_sorted)}
    
    comparison = []
    for feat in FEATURE_COLS:
        if feat in perm_rank and feat in shap_rank:
            comparison.append({
                'feature': feat,
                'perm_rank': perm_rank[feat] + 1,
                'shap_rank': shap_rank[feat] + 1,
                'rank_diff': abs(perm_rank[feat] - shap_rank[feat]),
            })
    
    comparison_df = pd.DataFrame(comparison).sort_values('perm_rank')
    
    print("\nRanking comparison (top 15 by permutation importance):")
    print("-" * 60)
    print(f"{'Feature':<35} {'Perm Rank':>10} {'SHAP Rank':>10} {'Diff':>6}")
    print("-" * 60)
    for _, row in comparison_df.head(15).iterrows():
        print(f"{row['feature']:<35} {row['perm_rank']:>10} {row['shap_rank']:>10} {row['rank_diff']:>6}")
    
    # Spearman correlation between rankings
    from scipy.stats import spearmanr
    corr, pval = spearmanr(comparison_df['perm_rank'], comparison_df['shap_rank'])
    print(f"\nSpearman rank correlation: {corr:.3f} (p-value: {pval:.2e})")

## 7 — Save artifacts

In [ ]:
# Save permutation importance results
perm_importance_payload = {
    "method": "permutation_importance",
    "model": "full_hybrid_ensemble",
    "n_test_samples": N_TEST_SAMPLES,
    "n_repeats": N_REPEATS,
    "scoring_metrics": ["mape", "r2"],
    "mape_importance": {
        row['feature']: {
            "mean": float(row['importance_mean']),
            "std": float(row['importance_std']),
            "rank": int(i + 1),
        }
        for i, (_, row) in enumerate(importance_mape.iterrows())
    },
    "r2_importance": {
        row['feature']: {
            "mean": float(row['importance_mean']),
            "std": float(row['importance_std']),
            "rank": int(i + 1),
        }
        for i, (_, row) in enumerate(importance_r2.iterrows())
    },
}

with open(OUT_DIR / "permutation_importance.json", "w") as f:
    json.dump(perm_importance_payload, f, indent=2)
print("Saved", OUT_DIR / "permutation_importance.json")

In [ ]:
# Save DataFrames as CSV for easy analysis
importance_mape.to_csv(OUT_DIR / "permutation_importance_mape.csv", index=False)
importance_r2.to_csv(OUT_DIR / "permutation_importance_r2.csv", index=False)
print("Saved", OUT_DIR / "permutation_importance_mape.csv")
print("Saved", OUT_DIR / "permutation_importance_r2.csv")

## 8 — Summary

Permutation importance provides **global** feature importance for the full hybrid ensemble by measuring how much model performance degrades when each feature is shuffled.

**Key differences from SHAP:**
- SHAP: Local (per-prediction) importance that sums to prediction
- Permutation: Global importance measuring predictive power loss

**Artifacts created:**
- `permutation_importance.json` — Full results with MAPE and R² metrics
- `permutation_importance_mape.csv` — Ranked features by MAPE importance
- `permutation_importance_r2.csv` — Ranked features by R² importance
- `permutation_importance_*.png` — Visualization plots

**Use cases:**
- Feature selection and pruning
- Identifying features the model relies on most
- Validating SHAP importance rankings
- Understanding global model behavior

In [ ]:
print("\n── Permutation Importance Artifacts ──")
for p in sorted(OUT_DIR.glob("permutation_importance*")):
    if p.is_file():
        print(f"  {p.name}: {p.stat().st_size/1024:,.1f} KB")